# Energy data → diverse prompt/completion JSONL for small-language-model training

Replaces the single-template generator in `data_conversion.ipynb`. That version emitted one prompt shape
(`date + T1 + RH_1 + T_out → Appliances`). Three inputs explain very little of the 10-minute appliance load,
so the same-looking prompt maps to wildly different answers. A model trained on it learns to invent a confident
number for any question. That is the false-positive behaviour this notebook is built to avoid.

**Design rules (derived from the papers in `MD files combined/` and from profiling `df_output.json`):**

1. **Grounded answers.** Wherever possible the prompt carries the evidence (a window of readings, a snapshot of
   sensors) and the completion is computed exactly from it, so the model learns to read the context, not recall numbers.
2. **Task diversity.** The papers frame this data as point/next-step forecasting, daily-energy regression (TSER),
   peak/level classification, robust-threshold anomaly detection, room comparison, weather relationships and
   feature relevance. Each becomes a task family with several paraphrased prompt templates and varied feature naming.
3. **Balanced binary labels.** Every yes/no task (spike? lights on? premise true?) is sampled 50/50 so "yes" is never
   the cheap default.
4. **Explicit "cannot be determined" answers.** The dataset has no occupancy labels, no per-appliance sub-metering, no
   fault labels and no tariff. Questions that need those get a refusal that says what *is* available. Each one is
   paired with an answerable contrast (for example, a cost question *with* a tariff in the prompt).
5. **Trap questions.** These cover `rv1`/`rv2` (random noise), `T6` (outdoor sensor, not a room),
   `Tdewpoint` ≠ `T_out`, Wh-per-10-min vs kWh vs W, and false premises ("why did it spike to 900 Wh?" when it didn't).
6. **No leakage.** The split is chronological. Level and percentile thresholds come from the train split only, and
   context windows never cross split boundaries.

**Not generated on purpose:** "predict the exact Appliances value from one sensor snapshot" (the old template).
The best published model for this data, a GBM, explains only about 57% of the test variance (Monash TSER archive, citing
Candanedo et al.). With three features the fit is far worse, so training on exact values teaches confident guessing.
Forecasting here always gives the model the recent load history, and states that its answers are estimates.

Input: `energydata_complete.csv` → output: `*.jsonl` (+ `.jsonl.zst` in the last section). Standard library only.

### 1. Configuration

In [1]:
from pathlib import Path

# Input / output (on Colab use '/content/energydata_complete.csv')
INPUT_CSV = Path('energydata_complete.csv')
OUTPUT_DIR = Path('.')
OUTPUT_STEM = 'energydata_sml'      # writes energydata_sml.jsonl (+ _train/_val/_test if WRITE_SPLIT_FILES)
WRITE_SPLIT_FILES = True
INCLUDE_METADATA = True             # adds "task", "split", "answerable", "label" next to prompt/completion

SEED = 42
# Examples generated per task family in the TRAIN split; val/test get the same per-task count scaled by
# their share of the timeline. Raise this to produce a larger corpus.
EXAMPLES_PER_TASK_TRAIN = 2500

# Chronological split boundaries (dataset spans 2016-01-11 17:00 → 2016-05-27 18:00)
TRAIN_END = '2016-04-20 23:59:59'
VAL_END = '2016-05-08 23:59:59'

### 2. Load and clean the CSV
The raw file has padded strings (`'  60'`), scientific notation (`' 6.5999999999999996e+00'`) and 17-digit float noise.
Values are parsed to floats here and rounded only at presentation time.

In [2]:
import csv
from datetime import datetime, timedelta

ROOM = {  # sensor -> human name (Candanedo et al. 2017; T6/RH_6 are OUTSIDE the building, north side)
    '1': 'kitchen', '2': 'living room', '3': 'laundry room', '4': 'office room', '5': 'bathroom',
    '6': 'outside the building (north side)', '7': 'ironing room', '8': 'teenager room', '9': 'parents room',
}
INDOOR_IDS = ['1', '2', '3', '4', '5', '7', '8', '9']

rows = []
with open(INPUT_CSV, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    header = [h.strip() for h in reader.fieldnames]
    for raw in reader:
        rec = {k.strip(): (v.strip() if isinstance(v, str) else v) for k, v in raw.items()}
        try:
            r = {'date': datetime.strptime(rec['date'], '%Y-%m-%d %H:%M:%S')}
            for k in header:
                if k != 'date':
                    r[k] = float(rec[k])
        except (ValueError, TypeError, KeyError):
            continue  # drop malformed rows rather than emitting garbage
        rows.append(r)

rows.sort(key=lambda r: r['date'])
# de-duplicate timestamps (keep first)
seen, clean = set(), []
for r in rows:
    if r['date'] not in seen:
        seen.add(r['date'])
        clean.append(r)
rows = clean

gaps = sum(1 for a, b in zip(rows, rows[1:]) if b['date'] - a['date'] != timedelta(minutes=10))
print(f'{len(rows)} rows, {rows[0]["date"]} -> {rows[-1]["date"]}, {gaps} gaps in the 10-minute grid')
print('columns:', header)

19735 rows, 2016-01-11 17:00:00 -> 2016-05-27 18:00:00, 0 gaps in the 10-minute grid
columns: ['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'rv1', 'rv2']


### 3. Derived fields, chronological split and train-only thresholds
`NSM` (seconds since midnight), weekday and week status are the time features used in the original study.
Consumption levels and spike thresholds are percentiles of the **train** split's `Appliances`. The column is heavily
right-skewed (median ≈ 60 Wh, p90 ≈ 200 Wh, max 1080 Wh), so means and fixed cut-offs would mislabel most intervals.

In [3]:
import random, statistics, math, json, hashlib
from collections import Counter, defaultdict

t_train_end = datetime.strptime(TRAIN_END, '%Y-%m-%d %H:%M:%S')
t_val_end = datetime.strptime(VAL_END, '%Y-%m-%d %H:%M:%S')

for i, r in enumerate(rows):
    d = r['date']
    r['idx'] = i
    r['NSM'] = d.hour * 3600 + d.minute * 60
    r['weekday'] = d.strftime('%A')
    r['week_status'] = 'weekend' if d.weekday() >= 5 else 'weekday'
    r['split'] = 'train' if d <= t_train_end else ('val' if d <= t_val_end else 'test')
    # a context window may only use rows of the same split; also break on grid gaps
    r['contig'] = (i > 0 and rows[i - 1]['split'] == r['split']
                   and d - rows[i - 1]['date'] == timedelta(minutes=10))

def run_start(i):
    """First index of the unbroken same-split run that contains row i."""
    while i > 0 and rows[i]['contig']:
        i -= 1
    return i

# precompute run starts once
_rs, start = [], 0
for i, r in enumerate(rows):
    if not r['contig']:
        start = i
    _rs.append(start)

def window_ok(i, back, fwd=0):
    """rows[i-back .. i+fwd] exist, are contiguous and lie in one split."""
    j = i + fwd
    return i - back >= 0 and j < len(rows) and _rs[j] == _rs[i] and i - back >= _rs[i]

def pct(sorted_vals, q):
    k = (len(sorted_vals) - 1) * q
    lo, hi = math.floor(k), math.ceil(k)
    return sorted_vals[lo] + (sorted_vals[hi] - sorted_vals[lo]) * (k - lo)

train_app = sorted(r['Appliances'] for r in rows if r['split'] == 'train')
Q = {q: pct(train_app, q) for q in (0.25, 0.5, 0.75, 0.9, 0.95, 0.99)}
LOW_MAX, HIGH_MIN, PEAK_MIN = Q[0.5], Q[0.9], Q[0.95]

def level(v):
    if v <= LOW_MAX:
        return 'low'
    if v < HIGH_MIN:
        return 'moderate'
    if v < PEAK_MIN:
        return 'high'
    return 'peak'

split_idx = defaultdict(list)
for r in rows:
    split_idx[r['split']].append(r['idx'])
print({s: len(v) for s, v in split_idx.items()})
print('train Appliances percentiles:', {k: round(v, 1) for k, v in Q.items()})
print('level bands: low <= %.0f < moderate < %.0f <= high < %.0f <= peak' % (LOW_MAX, HIGH_MIN, PEAK_MIN))
print('level distribution (train):', Counter(level(v) for v in train_app))

{'train': 14442, 'val': 2592, 'test': 2701}
train Appliances percentiles: {0.25: 50.0, 0.5: 60.0, 0.75: 100.0, 0.9: 210.0, 0.95: 340.0, 0.99: 590.0}
level bands: low <= 60 < moderate < 210 <= high < 340 <= peak
level distribution (train): Counter({'low': 8080, 'moderate': 4885, 'peak': 745, 'high': 732})


### 4. Presentation helpers
Feature naming is varied per example: raw column codes (`T1`), descriptive names (`kitchen temperature`), or
both. Layout is varied too (sentence, bullet list, `key: value`) so the model keys on meaning, not on one template.
Numbers are rounded to sensor precision, which removes the `19.890000000000001` artefacts from the old JSONL.

In [4]:
UNIT = {'T': '°C', 'RH': '%', 'T_out': '°C', 'Tdewpoint': '°C', 'RH_out': '%', 'Press_mm_hg': 'mm Hg',
        'Windspeed': 'm/s', 'Visibility': 'km', 'Appliances': 'Wh', 'lights': 'Wh', 'rv1': '', 'rv2': ''}

def unit(col):
    if col in UNIT:
        return UNIT[col]
    return UNIT[col.split('_')[0]] if col.startswith('RH_') else UNIT['T']

def nice_name(col):
    if col.startswith('RH_') and col != 'RH_out':
        return f'humidity in the {ROOM[col[3:]]}' if col[3:] != '6' else 'humidity outside the building (north side)'
    if col.startswith('T') and col[1:].isdigit():
        return f'temperature in the {ROOM[col[1:]]}' if col[1:] != '6' else 'temperature outside the building (north side)'
    return {'T_out': 'outdoor temperature (weather station)', 'Tdewpoint': 'outdoor dew point',
            'RH_out': 'outdoor humidity (weather station)', 'Press_mm_hg': 'atmospheric pressure',
            'Windspeed': 'wind speed', 'Visibility': 'visibility', 'Appliances': 'appliances energy use',
            'lights': 'lighting energy use', 'rv1': 'random variable rv1', 'rv2': 'random variable rv2'}[col]

def fmt(col, v):
    if col in ('Appliances', 'lights', 'Visibility'):
        return f'{v:.0f}'
    if col in ('rv1', 'rv2'):
        return f'{v:.2f}'
    return f'{v:.1f}'

def label(col, style):
    return {'code': col, 'name': nice_name(col), 'both': f'{nice_name(col)} ({col})'}[style]

def describe(r, cols, rng, layout=None, style=None):
    style = style or rng.choice(['code', 'name', 'both', 'name'])
    layout = layout or rng.choice(['sentence', 'bullets', 'kv'])
    parts = [(label(c, style), f'{fmt(c, r[c])} {unit(c)}'.strip()) for c in cols]
    if layout == 'sentence':
        return '; '.join(f'{k} is {v}' for k, v in parts) + '.'
    if layout == 'bullets':
        return '\n' + '\n'.join(f'- {k}: {v}' for k, v in parts)
    return ', '.join(f'{k}={v}' for k, v in parts)

def ts(d, rng=None):
    if rng is None or rng.random() < 0.6:
        return d.strftime('%Y-%m-%d %H:%M')
    return d.strftime('%A %d %B %Y at %H:%M')

def series(idxs, rng, col='Appliances'):
    """Render a run of readings as a timestamped list or a compact comma list."""
    if rng.random() < 0.5:
        return '\n' + '\n'.join(f'{rows[i]["date"].strftime("%H:%M")}  {fmt(col, rows[i][col])} Wh' for i in idxs)
    return ', '.join(fmt(col, rows[i][col]) for i in idxs) + ' (Wh, oldest first, 10-minute intervals)'

def pick(rng, *options):
    return rng.choice(options)

### 5. Task generators
Each generator takes `(rng, i)` for a row index `i` in the current split and returns
`(prompt, completion, meta)` or `None` if the row does not fit the task. `meta['label']` drives class balancing.

| family | grounded in prompt | basis in the papers |
|---|---|---|
| `forecast_next` / `forecast_hour` | last 3–12 readings | next-step / short-horizon load forecasting (LV load-forecasting review) |
| `window_stats` / `daily_profile` | the readings themselves | daily-energy regression (Monash TSER `AppliancesEnergy`) |
| `level_classify` | the value | peak / high-consumption classification |
| `spike_check` | 2-hour context | robust-threshold anomaly detection (anomaly / FDD papers) |
| `room_compare`, `outdoor_weather` | sensor snapshot | multi-zone temperature/humidity analysis, T6/Tdewpoint traps |
| `lights_activity` | lights readings | lights as an activity proxy, not occupancy (occupancy paper) |
| `time_features` | timestamp | NSM / week-status features of the original study |
| `units_cost` | value (+ tariff when given) | Wh/10 min ↔ W ↔ kWh confusion; cost needs a tariff |
| `noise_features` | rv1/rv2 | random variables included to test feature selection |
| `unanswerable` | — | occupancy, disaggregation, faults, out-of-range dates, missing sensors |
| `false_premise` | readings | prevents agreeing with a wrong claim in the question |

In [5]:
# ---- forecasting -----------------------------------------------------------------------------
def gen_forecast_next(rng, i):
    k = rng.choice([3, 6, 6, 12])
    if not window_ok(i, k, 1):
        return None
    hist = list(range(i - k + 1, i + 1))
    y = rows[i + 1]['Appliances']
    extra = ''
    if rng.random() < 0.5:
        cols = rng.sample(['T_out', 'RH_out', 'T1', 'T2', 'lights', 'Windspeed'], rng.randint(1, 3))
        extra = ' Current conditions: ' + describe(rows[i], cols, rng, layout='sentence')
    p = pick(rng,
             f'Here are the last {k} appliance energy readings of a house ending {ts(rows[i]["date"], rng)}: {series(hist, rng)}.{extra}\nForecast the appliances energy use for the next 10-minute interval.',
             f'Given these {k} consecutive 10-minute appliance readings: {series(hist, rng)}{extra}\nWhat is the expected energy use in the following 10 minutes?',
             f'Short-term load forecast. History (10-min steps, Wh): {series(hist, rng)}.{extra} Predict the next value.')
    c = (f'The next 10-minute reading is expected to be about {y:.0f} Wh, a {level(y)} level for this house. '
         f'This is a point forecast from the recent readings and it carries uncertainty: appliance load in this dataset '
         f'is spiky and right-skewed (median ≈ {LOW_MAX:.0f} Wh).')
    return p, c, {'label': level(y)}

def gen_forecast_hour(rng, i):
    if not window_ok(i, 11, 6):
        return None
    hist = list(range(i - 11, i + 1))
    tot = sum(rows[j]['Appliances'] for j in range(i + 1, i + 7))
    p = pick(rng,
             f'The last two hours of appliance energy readings (10-minute intervals) are: {series(hist, rng)}.\nEstimate the total appliances energy for the next hour.',
             f'Using the previous 12 readings {series(hist, rng)}, forecast the energy the appliances will use over the coming 60 minutes.')
    c = (f'The next hour is expected to total about {tot:.0f} Wh ({tot / 1000:.2f} kWh), which is the sum of six '
         f'10-minute intervals with an average of {tot / 6:.0f} Wh each. Treat this as an estimate; individual intervals can spike well above the average.')
    return p, c, {'label': None}

# ---- exact window statistics -------------------------------------------------------------------
def gen_window_stats(rng, i):
    k = rng.choice([4, 6, 8, 12, 18])
    if not window_ok(i, k - 1):
        return None
    idxs = list(range(i - k + 1, i + 1))
    vals = [rows[j]['Appliances'] for j in idxs]
    q = rng.choice(['total', 'mean', 'max', 'min', 'above'])
    head = f'Appliance energy readings from {ts(rows[idxs[0]]["date"])} to {ts(rows[idxs[-1]]["date"])} (10-minute intervals): {series(idxs, rng)}.\n'
    if q == 'total':
        s = sum(vals)
        return head + pick(rng, 'What is the total energy used?', 'How much energy did the appliances use in total over this period?'), \
            f'The total is {s:.0f} Wh ({s / 1000:.3f} kWh) across {k} readings.', {'label': None}
    if q == 'mean':
        m = statistics.mean(vals)
        return head + 'What is the average reading?', f'The average is {m:.1f} Wh per 10-minute interval (equivalent to an average power of about {m * 6:.0f} W).', {'label': None}
    if q in ('max', 'min'):
        f = max if q == 'max' else min
        v = f(vals)
        when = [rows[j]['date'].strftime('%H:%M') for j in idxs if rows[j]['Appliances'] == v]
        word = 'highest' if q == 'max' else 'lowest'
        return head + f'Which interval had the {word} consumption?', \
            f'The {word} reading is {v:.0f} Wh, at {", ".join(when)}.', {'label': None}
    thr = rng.choice([100, 150, 200, 300])
    n = sum(v > thr for v in vals)
    return head + f'How many intervals exceeded {thr} Wh?', \
        (f'{n} of the {k} intervals exceeded {thr} Wh.' if n else f'None of the {k} intervals exceeded {thr} Wh; the maximum was {max(vals):.0f} Wh.'), \
        {'label': 'yes' if n else 'no'}

# ---- daily profile (TSER-style) ---------------------------------------------------------------
DAY_ROWS = defaultdict(list)
for r in rows:
    DAY_ROWS[r['date'].date()].append(r['idx'])
FULL_DAYS = {d: ix for d, ix in DAY_ROWS.items() if len(ix) == 144 and len({rows[j]['split'] for j in ix}) == 1}

def gen_daily_profile(rng, i):
    d = rows[i]['date'].date()
    if d not in FULL_DAYS or rows[i]['date'].minute != 0:
        return None  # whole-hour anchors; the question type varies per anchor
    hourly = [sum(rows[j]['Appliances'] for j in FULL_DAYS[d][h * 6:(h + 1) * 6]) for h in range(24)]
    listing = ', '.join(f'{h:02d}h={v:.0f}' for h, v in enumerate(hourly))
    q = rng.choice(['total', 'peak', 'night', 'compare', 'range', 'range', 'count'])
    head = f'Hourly appliance energy (Wh) for {d.strftime("%A %Y-%m-%d")}: {listing}.\n'
    tot = sum(hourly)
    if q == 'total':
        return head + 'What was the total appliance energy for the day in kWh?', \
            f'The daily total is {tot / 1000:.2f} kWh ({tot:.0f} Wh, the sum of the 24 hourly values).', {'label': None}
    if q == 'peak':
        h = max(range(24), key=lambda x: hourly[x])
        return head + 'During which hour was consumption highest?', \
            f'The peak hour was {h:02d}:00-{h + 1:02d}:00 with {hourly[h]:.0f} Wh, which is {hourly[h] / tot * 100:.0f}% of the daily total of {tot:.0f} Wh.', {'label': None}
    if q == 'night':
        n = sum(hourly[0:6])
        return head + 'How much energy was used between midnight and 06:00, and what share of the day is that?', \
            f'Between 00:00 and 06:00 the appliances used {n:.0f} Wh, which is {n / tot * 100:.0f}% of the daily {tot:.0f} Wh.', {'label': None}
    if q == 'range':
        a = rng.randrange(0, 23)
        b = rng.randrange(a + 1, 24)
        s = sum(hourly[a:b + 1])
        return head + f'How much energy was used from {a:02d}:00 to {b + 1:02d}:00?', \
            f'From {a:02d}:00 to {b + 1:02d}:00 the appliances used {s:.0f} Wh ({s / 1000:.2f} kWh), the sum of {b - a + 1} hourly values.', {'label': None}
    if q == 'count':
        thr = rng.choice([500, 750, 1000, 1500])
        hrs = [h for h in range(24) if hourly[h] > thr]
        return head + f'In how many hours did consumption exceed {thr} Wh?', \
            (f'{len(hrs)} hours exceeded {thr} Wh: ' + ', '.join(f'{h:02d}h' for h in hrs) + '.' if hrs
             else f'No hour exceeded {thr} Wh; the highest was {max(hourly):.0f} Wh.'), {'label': 'yes' if hrs else 'no'}
    am, pm = sum(hourly[6:12]), sum(hourly[18:24])
    more = 'evening (18-24h)' if pm > am else 'morning (06-12h)' if am > pm else 'neither; they are equal'
    return head + 'Was more energy used in the morning (06-12h) or the evening (18-24h)?', \
        f'Morning used {am:.0f} Wh and evening used {pm:.0f} Wh, so more was used in the {more}.', {'label': None}

In [6]:
# ---- level classification --------------------------------------------------------------------
def gen_level_classify(rng, i):
    v = rows[i]['Appliances']
    lv = level(v)
    bands = (f'These bands are percentiles of the training period. For this house, readings up to {LOW_MAX:.0f} Wh are low (at or below the median), up to {HIGH_MIN:.0f} Wh moderate, '
             f'up to {PEAK_MIN:.0f} Wh high, and {PEAK_MIN:.0f} Wh or more is peak (top 5%).')
    if rng.random() < 0.5:
        p = pick(rng,
                 f'At {ts(rows[i]["date"], rng)} the appliances used {v:.0f} Wh in 10 minutes. How would you classify this consumption level?',
                 f'Classify a 10-minute appliance reading of {v:.0f} Wh as low, moderate, high or peak for this household.')
        return p, f'{v:.0f} Wh is {lv}. {bands}', {'label': lv}
    is_high = lv in ('high', 'peak')
    p = pick(rng,
             f'Is a reading of {v:.0f} Wh for one 10-minute interval a high-consumption interval for this house?',
             f'Appliances: {v:.0f} Wh at {ts(rows[i]["date"])}. Is that unusually high?')
    c = (f'Yes. {v:.0f} Wh is {lv}: it is at or above the 90th percentile ({HIGH_MIN:.0f} Wh). ' if is_high else
         f'No. {v:.0f} Wh is {lv}: it is below the 90th percentile ({HIGH_MIN:.0f} Wh). ') + bands
    return p, c, {'label': 'yes' if is_high else 'no'}

# ---- spike / anomaly check --------------------------------------------------------------------
def robust_stats(vals):
    med = statistics.median(vals)
    mad = statistics.median(abs(v - med) for v in vals)
    return med, max(1.4826 * mad, 10.0)   # floor: readings are quantised in 10 Wh steps

SPIKE_Z = 3.0  # residual z-score threshold; the anomaly papers start at 3 (≈99.7%) and tune 2.5-4.5

def gen_spike_check(rng, i):
    if not window_ok(i, 12):
        return None
    ctx = list(range(i - 12, i))
    vals = [rows[j]['Appliances'] for j in ctx]
    med, sd = robust_stats(vals)
    v = rows[i]['Appliances']
    z = (v - med) / sd
    spike = z >= SPIKE_Z and v >= HIGH_MIN
    p = pick(rng,
             f'Previous two hours of appliance readings: {series(ctx, rng)}.\nThe next reading is {v:.0f} Wh. Is this a consumption spike?',
             f'Context (12 readings, 10-min): {series(ctx, rng)}\nNew reading: {v:.0f} Wh. Should it be flagged as anomalous?',
             f'Check this reading for an anomaly. Recent history: {series(ctx, rng)}. Current: {v:.0f} Wh at {ts(rows[i]["date"])}.')
    basis = (f'The recent median is {med:.0f} Wh with a robust spread (1.4826 × MAD) of {sd:.0f} Wh, so the new reading is '
             f'{z:+.1f} robust standard deviations from typical.')
    if spike:
        c = (f'Yes, flag it. {basis} It also exceeds the house\'s 90th percentile ({HIGH_MIN:.0f} Wh). '
             'The data only shows total appliance energy, so it cannot tell which appliance caused the spike or whether it indicates a fault.')
    else:
        reason = (f'it is within {SPIKE_Z:g} robust standard deviations of recent behaviour' if z < SPIKE_Z
                  else f'although it deviates from the last two hours, it is still below the 90th percentile ({HIGH_MIN:.0f} Wh)')
        c = f'No. {basis} Do not flag it: {reason}.'
    return p, c, {'label': 'yes' if spike else 'no'}

In [7]:
# ---- room comparison (with T6 outdoor trap) ----------------------------------------------------
def gen_room_compare(rng, i):
    r = rows[i]
    kind = rng.choice(['T', 'RH'])
    ids = rng.sample(INDOOR_IDS, rng.randint(3, 5))
    include_t6 = rng.random() < 0.3
    if include_t6:
        ids.insert(rng.randrange(len(ids) + 1), '6')
    cols = [f'{kind}{x}' if kind == 'T' else f'RH_{x}' for x in ids]
    q = rng.choice(['max', 'min', 'diff'])
    what = 'temperature' if kind == 'T' else 'humidity'
    u = '°C' if kind == 'T' else '%'
    du = '°C' if kind == 'T' else 'percentage points'
    snap = describe(r, cols, rng)
    indoor = [c for c, x in zip(cols, ids) if x != '6']
    room = lambda c: ROOM[c[-1]]
    note = (f' {cols[ids.index("6")]} is not a room: that sensor is outside the building (north side), so it is excluded.'
            if include_t6 else '')
    if q in ('max', 'min'):
        f = max if q == 'max' else min
        best = f(indoor, key=lambda c: r[c])
        word = ('warmest' if q == 'max' else 'coldest') if kind == 'T' else ('most humid' if q == 'max' else 'driest')
        p = f'Sensor snapshot at {ts(r["date"], rng)}: {snap}\nWhich room is the {word}?'
        c = f'The {room(best)} ({best}) is the {word} room at {r[best]:.1f} {u}.{note}'
        return p, c, {'label': None}
    a, b = rng.sample(indoor, 2)
    d = r[a] - r[b]
    p = f'Snapshot: {snap}\nHow much higher is the {what} in the {room(a)} than in the {room(b)}?'
    rel = 'higher' if d > 0 else 'lower' if d < 0 else 'equal'
    c = (f'The {room(a)} is {abs(d):.1f} {du} {rel} than the {room(b)} ({r[a]:.1f} vs {r[b]:.1f} {u}).' if rel != 'equal'
         else f'They are equal at {r[a]:.1f} {u}.')
    return p, c, {'label': None}

# ---- outdoor weather (Tdewpoint vs T_out, station vs T6) ---------------------------------------
def gen_outdoor_weather(rng, i):
    r = rows[i]
    q = rng.choice(['spread', 'confuse_dew', 'indoor_vs_out', 'station_vs_t6', 'cold_heating'])
    if q == 'spread':
        s = r['T_out'] - r['Tdewpoint']
        p = f'Weather at {ts(r["date"], rng)}: {describe(r, ["T_out", "Tdewpoint", "RH_out"], rng)}\nWhat is the dew-point spread, and is the air close to saturation?'
        near = s <= 2.0
        c = (f'The spread is {s:.1f} °C (T_out − Tdewpoint). ' +
             (f'That is small, so the air is close to saturation, consistent with RH_out of {r["RH_out"]:.0f}%.' if near
              else f'That is not small, so the air is not saturated (RH_out {r["RH_out"]:.0f}%).'))
        return p, c, {'label': 'yes' if near else 'no'}
    if q == 'confuse_dew':
        p = f'The outdoor dew point is {r["Tdewpoint"]:.1f} °C. So the outside temperature is {r["Tdewpoint"]:.1f} °C, right?'
        c = ('No. Tdewpoint is the dew point, not the air temperature. The air temperature is T_out, which is always at or above the dew point, '
             f'so the outside temperature was at least {r["Tdewpoint"]:.1f} °C. The exact value needs the T_out reading, which was not given.')
        return p, c, {'label': 'no'}
    if q == 'indoor_vs_out':
        x = rng.choice(INDOOR_IDS)
        d = r[f'T{x}'] - r['T_out']
        p = f'{describe(r, [f"T{x}", "T_out"], rng)}\nHow much warmer is the {ROOM[x]} than outdoors?'
        c = (f'The {ROOM[x]} is {d:.1f} °C warmer than outdoors.' if d >= 0 else
             f'It is not warmer: the {ROOM[x]} is {-d:.1f} °C cooler than outdoors.')
        return p, c, {'label': None}
    if q == 'cold_heating':
        cold = r['T_out'] < 5
        p = f'Outdoor temperature is {r["T_out"]:.1f} °C and the appliances used {r["Appliances"]:.0f} Wh. ' + \
            ('It is cold, so is the high appliance use caused by heating?' if cold else 'Does the outdoor temperature explain this appliance reading?')
        c = ('No. Heating is not part of the Appliances measurement. The house was heated with a wood stove, which is not in the data, '
             'so outdoor temperature cannot be used to explain this reading. '
             f'{r["Appliances"]:.0f} Wh is a {level(r["Appliances"])} level for this house.')
        return p, c, {'label': 'no'}
    p = f'{describe(r, ["T6", "T_out"], rng, style="both")}\nWhy do these two outdoor temperatures differ?'
    c = (f'T6 ({r["T6"]:.1f} °C) is measured by a sensor on the north side of the building. T_out ({r["T_out"]:.1f} °C) comes from a '
         'nearby airport weather station (Chièvres). The location, sun exposure and the building itself make them differ, '
         f'here by {abs(r["T6"] - r["T_out"]):.1f} °C.')
    return p, c, {'label': None}

# ---- lights: activity proxy, not occupancy -----------------------------------------------------
def gen_lights_activity(rng, i):
    k = rng.choice([3, 6])
    if not window_ok(i, k - 1):
        return None
    idxs = list(range(i - k + 1, i + 1))
    lv = [rows[j]['lights'] for j in idxs]
    on = any(v > 0 for v in lv)
    hist = ', '.join(f'{v:.0f}' for v in lv)
    span_txt = f'{rows[idxs[0]]["date"]:%Y-%m-%d %H:%M}-{rows[idxs[-1]]["date"]:%H:%M}'
    if rng.random() < 0.6:
        p = pick(rng, f'Lighting energy {span_txt} (Wh per 10 min): {hist}. Were the lights in use?',
                 f'lights readings for {span_txt}: {hist}. Was any lighting switched on?')
        c = (f'Yes. {sum(v > 0 for v in lv)} of the {k} intervals show lighting consumption (up to {max(lv):.0f} Wh).' if on else
             f'No. All {k} intervals show 0 Wh of lighting energy.')
        return p, c, {'label': 'yes' if on else 'no'}
    p = f'Lighting energy {span_txt} (Wh per 10 min): {hist}. How many people were at home?'
    c = ('That cannot be determined. The dataset has no occupancy sensor or people count. ' +
         ('Lighting use suggests some activity in the house, but it cannot say how many people were home, '
          'and lights can be left on in an empty house.' if on else
          'Zero lighting energy does not mean the house was empty, for example during daylight or while people are asleep.'))
    return p, c, {'label': 'unanswerable'}

# ---- time features ------------------------------------------------------------------------------
def gen_time_features(rng, i):
    r = rows[i]
    q = rng.choice(['nsm', 'week', 'dow'])
    if q == 'nsm':
        return f'How many seconds after midnight is the timestamp {ts(r["date"])} (the NSM feature)?', \
            f'{r["date"].strftime("%H:%M")} is {r["NSM"]} seconds after midnight ({r["date"].hour} h × 3600 + {r["date"].minute} min × 60).', {'label': None}
    if q == 'week':
        we = r['week_status'] == 'weekend'
        return f'Is {r["date"].strftime("%Y-%m-%d")} a weekend day?', \
            f'{"Yes" if we else "No"}, {r["date"].strftime("%Y-%m-%d")} is a {r["weekday"]}, so its week status is {r["week_status"]}.', {'label': 'yes' if we else 'no'}
    return f'What day of the week was {r["date"].strftime("%Y-%m-%d")}?', f'{r["date"].strftime("%Y-%m-%d")} was a {r["weekday"]}.', {'label': None}

In [8]:
# ---- units and cost --------------------------------------------------------------------------
def gen_units_cost(rng, i):
    v = rows[i]['Appliances']
    q = rng.choice(['power', 'kwh', 'cost_given', 'cost_missing'])
    if q == 'power':
        return pick(rng, f'The appliances used {v:.0f} Wh in a 10-minute interval. What was the average power draw?',
                    f'A 10-minute appliance reading is {v:.0f} Wh. Convert it to average watts.'), \
            f'{v:.0f} Wh over 10 minutes (1/6 h) is an average of {v * 6:.0f} W ({v * 6 / 1000:.2f} kW).', {'label': None}
    if q == 'kwh':
        return f'If {v:.0f} Wh were used every 10 minutes for a whole day, how many kWh would that be?', \
            f'144 intervals × {v:.0f} Wh = {v * 144:.0f} Wh = {v * 144 / 1000:.2f} kWh.', {'label': None}
    if q == 'cost_given':
        price = rng.choice([0.15, 0.22, 0.28, 0.31, 0.35])
        cost = v / 1000 * price
        return f'Electricity costs €{price:.2f} per kWh. At {ts(rows[i]["date"])} the appliances used {v:.0f} Wh in 10 minutes. What did that interval cost?', \
            f'{v:.0f} Wh = {v / 1000:.3f} kWh, and {v / 1000:.3f} × €{price:.2f} = €{cost:.4f}.', {'label': 'answerable'}
    return f'The appliances used {v:.0f} Wh between {ts(rows[i]["date"])} and 10 minutes later. How much did that cost?', \
        ('That cannot be calculated from the data: the dataset contains no electricity tariff or price. '
         f'Give a price per kWh and multiply it by {v / 1000:.3f} kWh to get the cost.'), {'label': 'unanswerable'}

# ---- noise features (rv1 / rv2) -----------------------------------------------------------------
def gen_noise_features(rng, i):
    r = rows[i]
    p = pick(rng,
             f'rv1 is {r["rv1"]:.2f} and rv2 is {r["rv2"]:.2f}. How do these values affect appliance energy use?',
             f'At {ts(r["date"])} rv1={r["rv1"]:.2f}. Does a higher rv1 mean higher consumption?',
             f'Should rv1 and rv2 be used as features to predict appliance energy? Current values: {r["rv1"]:.2f}, {r["rv2"]:.2f}.')
    c = ('They have no effect. rv1 and rv2 are random variables that the dataset authors added on purpose to test feature selection. '
         'They carry no information about consumption (rv2 is an exact copy of rv1), so they should be excluded '
         'from a model and never used to explain a reading.')
    return p, c, {'label': 'no'}

In [9]:
# ---- unanswerable / insufficient information ---------------------------------------------------
FIRST, LAST = rows[0]['date'], rows[-1]['date']
COVER = f'The dataset covers {FIRST:%Y-%m-%d %H:%M} to {LAST:%Y-%m-%d %H:%M} at 10-minute resolution'

def gen_unanswerable(rng, i):
    r = rows[i]
    v = r['Appliances']
    kind = rng.choice(['occupancy', 'appliance', 'fault', 'no_context', 'out_of_range', 'missing_sensor', 'long_horizon'])
    if kind == 'occupancy':
        p = pick(rng, f'At {ts(r["date"], rng)}: {describe(r, ["Appliances", "lights", "T1", "RH_1"], rng)}\nWas anyone home?',
                 f'Given {describe(r, ["T2", "RH_2", "lights"], rng)} how many occupants are in the living room?')
        c = ('That cannot be determined from this data. It has no occupancy, motion or CO2 sensor. Temperature, humidity and energy '
             'can change for many reasons, such as weather, heating schedules or appliances on timers, so any answer would be a guess.')
    elif kind == 'appliance':
        p = pick(rng, f'The appliances used {v:.0f} Wh at {ts(r["date"])}. Which appliance was running?',
                 f'Break down this {v:.0f} Wh reading into fridge, washing machine and oven consumption.')
        c = (f'That cannot be determined. {v:.0f} Wh is a single aggregate meter reading for all appliances. The dataset has no per-appliance '
             'sub-metering, so separating it into individual devices (load disaggregation) is not possible without device-level data or signatures.')
    elif kind == 'fault':
        p = f'{describe(r, ["T1", "T2", "T3", "Appliances"], rng)}\nIs the heating system faulty?'
        c = ('That cannot be diagnosed from this data. It has no HVAC operating signals (setpoints, supply/return temperatures, valve or '
             'fan states) and no fault labels, and the house was heated with wood rather than a metered HVAC system. These readings only show conditions and total appliance energy. '
             'A sustained unusual pattern can be flagged for inspection, but not attributed to a fault.')
    elif kind == 'no_context':
        p = pick(rng, f'What was the appliances energy use on {ts(r["date"])}?',
                 f'Tell me the exact kitchen temperature recorded at {ts(r["date"])}.')
        c = ('I cannot give a recorded value without the data. No reading for that timestamp was included in the question. '
             'Provide the readings (or the row for that time) and I can report or analyse them.')
    elif kind == 'out_of_range':
        d = rng.choice([FIRST - timedelta(days=rng.randint(10, 400)), LAST + timedelta(days=rng.randint(10, 400))])
        p = f'What was the appliances energy consumption on {d:%Y-%m-%d} at {d:%H}:00 according to the dataset?'
        c = f'That date is outside the dataset. {COVER}, so there is no recorded value for {d:%Y-%m-%d}.'
    elif kind == 'missing_sensor':
        give, ask = rng.sample(INDOOR_IDS, 2)
        p = f'The {ROOM[give]} temperature is {r["T" + give]:.1f} °C and its humidity is {r["RH_" + give]:.1f}%. What is the humidity in the {ROOM[ask]}?'
        c = (f'That cannot be determined from what was given. Only the {ROOM[give]} readings were provided, and humidity differs '
             f'between rooms. A reading from the {ROOM[ask]} sensor (RH_{ask}) is needed.')
    else:
        p = f'The appliances used {v:.0f} Wh at {ts(r["date"])}. Predict the total energy for the whole next month.'
        c = ('One 10-minute reading is not enough to forecast a month. Appliance load varies strongly by time of day, weekday and '
             'season. A monthly estimate needs at least several weeks of history (for example daily totals) and ideally weather forecasts.')
    return p, c, {'label': 'unanswerable', 'kind': kind}

# ---- false premise (balanced with true premise) ------------------------------------------------
def gen_false_premise(rng, i):
    if not window_ok(i, 5):
        return None
    idxs = list(range(i - 5, i + 1))
    vals = [rows[j]['Appliances'] for j in idxs]
    mx = max(vals)
    true_case = rng.random() < 0.5 and mx >= HIGH_MIN
    if true_case:
        claim = mx
    else:
        claim = round(rng.uniform(max(mx * 2, HIGH_MIN + 100), 1100), -1)
    p = pick(rng,
             f'Readings for the hour ending {ts(rows[i]["date"])}: {series(idxs, rng)}.\nWhy did consumption jump to {claim:.0f} Wh during this hour?',
             f'In this hour {series(idxs, rng)} the appliances peaked at {claim:.0f} Wh. What caused it?')
    if true_case:
        c = (f'The readings do reach {mx:.0f} Wh, a {level(mx)} level for this house. The cause cannot be identified: the meter records total '
             'appliance energy only, so it cannot tell which device was switched on.')
        return p, c, {'label': 'premise_true'}
    c = (f'The premise is incorrect. None of these readings reaches {claim:.0f} Wh; the highest is {mx:.0f} Wh '
         f'({level(mx)} for this house). There is no {claim:.0f} Wh jump to explain.')
    return p, c, {'label': 'premise_false'}

TASKS = {
    'forecast_next': gen_forecast_next, 'forecast_hour': gen_forecast_hour, 'window_stats': gen_window_stats,
    'daily_profile': gen_daily_profile, 'level_classify': gen_level_classify, 'spike_check': gen_spike_check,
    'room_compare': gen_room_compare, 'outdoor_weather': gen_outdoor_weather, 'lights_activity': gen_lights_activity,
    'time_features': gen_time_features, 'units_cost': gen_units_cost, 'noise_features': gen_noise_features,
    'unanswerable': gen_unanswerable, 'false_premise': gen_false_premise,
}
# relative weights (1.0 = EXAMPLES_PER_TASK_TRAIN)
WEIGHT = {'forecast_next': 1.5, 'forecast_hour': 0.8, 'window_stats': 1.0, 'daily_profile': 1.0, 'level_classify': 1.0,
          'spike_check': 1.2, 'room_compare': 1.0, 'outdoor_weather': 0.6, 'lights_activity': 0.8, 'time_features': 0.4,
          'units_cost': 0.8, 'noise_features': 0.2, 'unanswerable': 1.2, 'false_premise': 0.8}
BALANCED = {'spike_check', 'level_classify', 'lights_activity', 'false_premise'}  # equalise their labels

### 6. Generate, balance and de-duplicate
For tasks in `BALANCED`, candidates are bucketed by label and each bucket is sampled to the same size. That gives
50/50 yes/no for spike detection, for example, even though real spikes are rare. Prompts are de-duplicated
across the whole corpus, and any prompt that maps to two different completions is dropped (the main source of
the "same question, random answer" problem in the original JSONL).

In [10]:
SYSTEM_HINT = None  # e.g. 'You are an energy-data assistant. Answer only from the data provided.'  (prepended if set)

def generate(split, n_target, task, fn, rng):
    pool = split_idx[split][:]
    rng.shuffle(pool)
    by_label = defaultdict(list)
    for i in pool:
        out = fn(rng, i)
        if out is None:
            continue
        p, c, meta = out
        by_label[meta.get('label')].append((p, c, meta))
        if task not in BALANCED and sum(map(len, by_label.values())) >= n_target:
            break
        if task in BALANCED and min((len(v) for v in by_label.values()), default=0) >= n_target and len(by_label) > 1:
            break
    if task in BALANCED and len(by_label) > 1:
        per = min(n_target // len(by_label), min(len(v) for v in by_label.values()))
        items = [x for v in by_label.values() for x in v[:per]]
    else:
        items = [x for v in by_label.values() for x in v][:n_target]
    rng.shuffle(items)
    return items

rng = random.Random(SEED)
span = {s: len(v) for s, v in split_idx.items()}
records, prompt_to_completion, conflicts, conflict_tasks = [], {}, set(), Counter()
for split in ('train', 'val', 'test'):
    scale = span[split] / span['train']
    for task, fn in TASKS.items():
        n = max(20, int(EXAMPLES_PER_TASK_TRAIN * WEIGHT[task] * scale))
        for p, c, meta in generate(split, n, task, fn, rng):
            if SYSTEM_HINT:
                p = SYSTEM_HINT + '\n\n' + p
            key = hashlib.sha1(p.encode()).hexdigest()
            if key in prompt_to_completion:
                if prompt_to_completion[key] != c:
                    conflicts.add(key)
                    conflict_tasks[task] += 1
                continue
            prompt_to_completion[key] = c
            rec = {'prompt': p, 'completion': c}
            if INCLUDE_METADATA:
                rec.update({'task': task, 'split': split,
                            'answerable': meta.get('label') != 'unanswerable', 'label': meta.get('label')})
            records.append((key, rec))

records = [rec for key, rec in records if key not in conflicts]
print(f'{len(records)} examples, {len(conflicts)} conflicting prompts removed', dict(conflict_tasks))

33916 examples, 24 conflicting prompts removed {'forecast_next': 33}


### 7. Sanity checks: distribution, balance, leakage, length

In [11]:
by_split = Counter(r.get('split') for r in records)
by_task = Counter((r.get('task'), r.get('split')) for r in records)
print('per split:', dict(by_split))
print(f'{"task":18} {"train":>6} {"val":>6} {"test":>6}')
for t in TASKS:
    print(f'{t:18} {by_task[(t, "train")]:>6} {by_task[(t, "val")]:>6} {by_task[(t, "test")]:>6}')

print('\nlabel balance (train):')
for t in sorted(BALANCED | {'unanswerable', 'units_cost'}):
    print(f'  {t:16}', dict(Counter(r['label'] for r in records if r['task'] == t and r['split'] == 'train')))

unans = sum(not r['answerable'] for r in records)
print(f'\nunanswerable share: {unans / len(records):.1%}')

# leakage: no prompt text shared across splits
split_of = defaultdict(set)
for r in records:
    split_of[r['prompt']].add(r['split'])
assert not any(len(s) > 1 for s in split_of.values()), 'prompt appears in more than one split'

lens = sorted(len(r['prompt']) + len(r['completion']) for r in records)
print(f'chars per example: median {lens[len(lens) // 2]}, p95 {lens[int(len(lens) * .95)]}, max {lens[-1]}')

for t in TASKS:  # one sample per task
    ex = next(r for r in records if r['task'] == t)
    print(f'\n=== {t} ===\nPROMPT: {ex["prompt"]}\nCOMPLETION: {ex["completion"]}')

per split: {'train': 25027, 'val': 4342, 'test': 4547}
task                train    val   test
forecast_next        3703    662    685
forecast_hour        2000    358    374
window_stats         2500    448    467
daily_profile        1299    231    240
level_classify       1248    153    191
spike_check          1624    262    250
room_compare         2500    448    467
outdoor_weather      1295    219    232
lights_activity      1998    357    372
time_features         504     98    105
units_cost           1127    194    199
noise_features        496     88     91
unanswerable         2733    466    500
false_premise        2000    358    374

label balance (train):
  false_premise    {'premise_true': 1000, 'premise_false': 1000}
  level_classify   {'moderate': 205, 'yes': 234, 'high': 195, 'low': 190, 'no': 194, 'peak': 230}
  lights_activity  {'no': 666, 'yes': 666, 'unanswerable': 666}
  spike_check      {'no': 812, 'yes': 812}
  unanswerable     {'unanswerable': 2733}
  units_c

### 8. Write JSONL

In [12]:
def write_jsonl(path, recs):
    with open(path, 'w', encoding='utf-8') as f:
        for rec in recs:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    return path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
written = [write_jsonl(OUTPUT_DIR / f'{OUTPUT_STEM}.jsonl', [r for r in records])]
if WRITE_SPLIT_FILES:
    for s in ('train', 'val', 'test'):
        written.append(write_jsonl(OUTPUT_DIR / f'{OUTPUT_STEM}_{s}.jsonl', [r for r in records if r.get('split') == s]))

for p in written:
    with open(p, encoding='utf-8') as f:
        n = sum(1 for line in f if json.loads(line))
    print(f'{p}  {n} lines  {p.stat().st_size / 1e6:.2f} MB')

energydata_sml.jsonl  33916 lines  15.91 MB
energydata_sml_train.jsonl  25027 lines  11.76 MB
energydata_sml_val.jsonl  4342 lines  2.03 MB
energydata_sml_test.jsonl  4547 lines  2.12 MB


### 9. Compress JSONL → `.jsonl.zst`
Uses Zstandard: the stdlib `compression.zstd` on Python ≥ 3.14, otherwise the `zstandard` package (installed
automatically if `AUTO_INSTALL` is on). If neither is available it falls back to `.jsonl.xz` (stdlib `lzma`).
Each archive is decompressed again and compared byte-for-byte before the result is reported.
Hugging Face `datasets` reads both `.jsonl.zst` and `.jsonl.xz` directly with `load_dataset('json', data_files=...)`.

In [13]:
import importlib, subprocess, sys, lzma

ZSTD_LEVEL = 19          # 1 (fast) .. 22 (max); 19 is a good size/speed trade-off for one-off packaging
AUTO_INSTALL = True
REMOVE_UNCOMPRESSED = False

def zstd_backend():
    try:
        from compression import zstd as z            # Python 3.14+
        return 'stdlib', z
    except ImportError:
        pass
    try:
        return 'zstandard', importlib.import_module('zstandard')
    except ImportError:
        if AUTO_INSTALL:
            r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'zstandard'], capture_output=True)
            if r.returncode == 0:
                importlib.invalidate_caches()
                return 'zstandard', importlib.import_module('zstandard')
    return None, None

backend, zmod = zstd_backend()
print('compression backend:', backend or 'lzma (xz fallback)')

def compress(src):
    data = src.read_bytes()
    if backend == 'stdlib':
        dst = src.with_suffix('.jsonl.zst')
        dst.write_bytes(zmod.compress(data, level=ZSTD_LEVEL))
        back = zmod.decompress(dst.read_bytes())
    elif backend == 'zstandard':
        dst = src.with_suffix('.jsonl.zst')
        with open(src, 'rb') as fi, open(dst, 'wb') as fo:   # streaming: fine for multi-GB files
            zmod.ZstdCompressor(level=ZSTD_LEVEL, threads=-1).copy_stream(fi, fo)
        with open(dst, 'rb') as fi:
            back = zmod.ZstdDecompressor().stream_reader(fi).read()
    else:
        dst = src.with_suffix('.jsonl.xz')
        with lzma.open(dst, 'wb', preset=9) as fo:
            fo.write(data)
        with lzma.open(dst, 'rb') as fi:
            back = fi.read()
    assert back == data, f'round-trip mismatch for {dst}'
    print(f'{src.name}: {len(data) / 1e6:.2f} MB -> {dst.name}: {dst.stat().st_size / 1e6:.2f} MB '
          f'(ratio {len(data) / dst.stat().st_size:.1f}x, verified)')
    if REMOVE_UNCOMPRESSED:
        src.unlink()
    return dst

archives = [compress(p) for p in written]

compression backend: zstandard


energydata_sml.jsonl: 15.91 MB -> energydata_sml.jsonl.zst: 0.54 MB (ratio 29.6x, verified)


energydata_sml_train.jsonl: 11.76 MB -> energydata_sml_train.jsonl.zst: 0.39 MB (ratio 29.8x, verified)


energydata_sml_val.jsonl: 2.03 MB -> energydata_sml_val.jsonl.zst: 0.08 MB (ratio 25.1x, verified)


energydata_sml_test.jsonl: 2.12 MB -> energydata_sml_test.jsonl.zst: 0.08 MB (ratio 25.6x, verified)


In [14]:
# Reading the compressed file back for training (streaming, no full decompression to disk)
import io

def iter_jsonl_compressed(path):
    path = Path(path)
    if path.suffix == '.zst':
        if backend == 'stdlib':
            fh = zmod.open(path, 'rb')
        else:
            fh = zmod.ZstdDecompressor().stream_reader(open(path, 'rb'))
    else:
        fh = lzma.open(path, 'rb')
    with io.TextIOWrapper(fh, encoding='utf-8') as text:
        for line in text:
            yield json.loads(line)

first = next(iter_jsonl_compressed(archives[0]))
print(json.dumps(first, indent=2, ensure_ascii=False))

{
  "prompt": "Short-term load forecast. History (10-min steps, Wh): \n22:00  90 Wh\n22:10  60 Wh\n22:20  60 Wh\n22:30  50 Wh\n22:40  60 Wh\n22:50  50 Wh. Predict the next value.",
  "completion": "The next 10-minute reading is expected to be about 60 Wh, a low level for this house. This is a point forecast from the recent readings and it carries uncertainty: appliance load in this dataset is spiky and right-skewed (median ≈ 60 Wh).",
  "task": "forecast_next",
  "split": "train",
  "answerable": true,
  "label": "low"
}
